In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# **Loading Dataset**

In [ ]:
df = pd.read_csv('/content/nutrition_deficiency_dataset2.csv')
df.head()

In [ ]:
print(df.info())

In [ ]:
# Checking Null values
df.isnull().sum()

In [ ]:
df['Deficiency'] = df['Deficiency'].fillna("None")

# **Encoding**

In [ ]:
df['Deficiency'] = df['Deficiency'].map({'None' : 0, 'Iron' : 1, 'VitaminB12' : 2, 'VitaminD' : 3, 'Calcium' : 4})

In [ ]:
plt.figure(figsize = [8,4])
sns.heatmap( df.corr(), annot = True)

# **Data Split**

In [ ]:
X = df.drop('Deficiency', axis = 1)
Y = df['Deficiency']

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split( X, Y, test_size = 0.2, random_state = 42, stratify = Y )

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.fit_transform(X_test)

# **Selecting the Best Model**

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(),
    "KNN": KNeighborsClassifier(),
    "SVM": SVC(),
    "Gradient Boosting": GradientBoostingClassifier(),
    "AdaBoost": AdaBoostClassifier()
}

In [ ]:
for name, model in models.items():
    model.fit(X_train_scaled, Y_train)
    pred = model.predict(X_test_scaled)
    acc = accuracy_score(Y_test, pred)

    print(name, "Accuracy:", acc)

# **Using Support Vector Machine**

In [ ]:
param_grid = {
    'C': [0.1, 1, 10],
    'kernel': ['linear', 'rbf','poly'],
    'gamma': ['scale', 'auto']
}

grid = GridSearchCV ( SVC(), param_grid, cv = 5 )
grid.fit(X_train_scaled, Y_train)

print("Best Parameters:", grid.best_params_)
print("Best Accuracy:", grid.best_score_)

# **Using Random Forest Classifier**

In [ ]:
# Hyperparameter Tuning
parameter = {
    'n_estimators': [100, 200],
    'max_depth': [4, 6, 8],
    'min_samples_split': [10, 20],
    'min_samples_leaf': [5, 10],
    'max_features': ['sqrt']
}

In [ ]:
rf = RandomForestClassifier( random_state = 42)

In [ ]:
grid_search = GridSearchCV(
    estimator = rf,
    param_grid = parameter,
    n_jobs = -1,
    verbose = 2,
    cv = 5,
    scoring='accuracy',
)

In [ ]:
grid_search.fit(X_train_scaled , Y_train)

In [ ]:
best_rf = grid_search.best_estimator_

In [ ]:
y_pred = best_rf.predict(X_test_scaled)

In [ ]:
print("Best Parameters:", grid_search.best_params_)
print("Best Accuracy:", accuracy_score(Y_test , y_pred))

# **Predictions**

In [ ]:
df.head(1)

In [ ]:
input = np.array([[25, 1, 22, 12, 5, 18, 900, 1, 1, 3, 1]])
input_scaled = scaler.transform(input)
predictions = best_rf.predict(input_scaled)
print(predictions)

# **Model Evaluation Metrics**

In [ ]:
print(classification_report(Y_test, y_pred))

In [ ]:
print("Accuracy Score :",accuracy_score(Y_test , y_pred))

In [ ]:
import joblib
from google.colab import files

In [ ]:
joblib.dump(best_rf, 'Smart_Nutrition_Prediction.pkl')
joblib.dump(scaler, 'Scale.pkl')

In [ ]:
files.download('Smart_Nutrition_Prediction.pkl')
files.download('Scale.pkl')